# Семинар №11. ETL-пайплайны и best practices батчинга

## Как устроен ноутбук

1. Каждая тема идёт по схеме: **теория → демо → упражнение**.
2. Ячейки с комментарием `# УПРАЖНЕНИЕ N` — для самостоятельной работы; там стоят `TODO`.
3. Выполняйте ячейки **по порядку**: состояние таблиц в БД важно (мы переиспользуем одни и те же таблицы).
4. Сброс: `Kernel → Restart & Run All`.

## Окружение

- PostgreSQL 16 слушает `localhost:5432` **внутри контейнера**.
- Учётные данные БД: пользователь `postgres`, **пароль не требуется** (trust auth для локальных подключений), база `etl_db`.
- База `etl_db` создаётся автоматически в ячейке настройки, если её ещё нет.
- Файл `events.csv` (1 000 000 строк телеметрии IoT, ~88 МБ) лежит **рядом с ноутбуком**.
- Jupyter-пароль: `student`.

## План

1. Простейший ETL-пайплайн (Extract → Transform → Load) с чанкованием.
2. Best practices батчинга: что происходит при INSERT, «гонка» способов вставки.
3. Транзакция на батч vs одна большая транзакция.
4. Влияние индексов и UNLOGGED-таблиц на скорость загрузки.
5. Чек-лист батчинга и итоги.


In [ ]:
# Проверка окружения и общая конфигурация
import sys
print("Python:", sys.version.split()[0])

import psycopg, pandas, numpy
print("psycopg     :", psycopg.__version__)
print("pandas      :", pandas.__version__)
print("numpy       :", numpy.__version__)

from pathlib import Path
import pandas as pd

# Параметры подключения (внутри контейнера localhost = этот же контейнер).
# PostgreSQL настроен с trust-аутентификацией для локальных подключений,
# поэтому пароль НЕ требуется и НЕ передаётся.
DB = {
    "host":   "localhost",
    "port":   5432,
    "dbname": "etl_db",
    "user":   "postgres",
}

PSYCOPG_CONN = (
    f"host={DB['host']} port={DB['port']} dbname={DB['dbname']} "
    f"user={DB['user']}"
)
print("\npsycopg DSN:", PSYCOPG_CONN)

# Проверка events.csv (лежит рядом с ноутбуком)
CSV = Path("events.csv")
print("\nevents.csv exists:", CSV.exists())
if CSV.exists():
    print("events.csv size MB:", round(CSV.stat().st_size / 1e6, 1))

# --- Создаём etl_db, если её ещё нет (подключаемся к служебной postgres) ---
ADMIN_CONN = (
    f"host={DB['host']} port={DB['port']} dbname=postgres user={DB['user']}"
)
with psycopg.connect(ADMIN_CONN, autocommit=True) as admin:
    with admin.cursor() as cur:
        cur.execute("SELECT 1 FROM pg_database WHERE datname = %s", (DB["dbname"],))
        if cur.fetchone() is None:
            # имя БД нельзя параметризовать через %s; экранируем кавычки
            safe_name = DB["dbname"].replace('"', '""')
            cur.execute(f'CREATE DATABASE "{safe_name}"')
            print(f"database {DB['dbname']} created")
        else:
            print(f"database {DB['dbname']} already exists")

# --- Подключаемся к etl_db и создаём raw_events, если её ещё нет ---
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS raw_events (
                event_id   uuid,
                ts         timestamp,
                device_id  integer,
                region     text,
                event_type text,
                value      numeric,
                status     text
            )
        """)
    conn.commit()
    with conn.cursor() as cur:
        cur.execute("SELECT version()")
        print("\nPG version:", cur.fetchone()[0].split(" on ")[0])
        cur.execute("SELECT count(*) FROM raw_events")
        print("raw_events rows:", cur.fetchone()[0])


## 1. Простейший ETL-пайплайн (Extract → Transform → Load)

**ETL** = Extract, Transform, Load: извлечение из источника → очистка/дедупликация → загрузка в целевую БД. В нашей задаче источник — `events.csv`, приёмник — таблица `raw_events` в PostgreSQL.

### Зачем чанкование

Файл `events.csv` — ~88 МБ, его можно прочитать целиком. Но промышленные выгрузки — десятки гигабайт, и в RAM они не помещаются. `pandas.read_csv(chunksize=N)` возвращает **итератор** по чанкам: в памяти одновременно живёт один чанк, память `O(chunk)`, а не `O(dataset)`.

> Правило из практики Data Engineering: размер чанка подбирают так, чтобы он занимал 100–500 МБ в памяти после парсинга типов. CSV-текст «раздувается» при превращении в объекты в 3–5 раз.

### Ключевые конструкции

- **`pd.read_csv(chunksize=...)`** — итератор по чанкам.
- **`df.dropna(subset=[...])`** и **`df.drop_duplicates(subset=[...])`** — векторные операции pandas для очистки (быстро).
- **`df.itertuples(index=False)`** — обход строк DataFrame кортежами; в разы быстрее `iterrows()`, а `index=False` отбрасывает индекс, чтобы кортеж совпадал по arity со списком колонок в `COPY`.
- **`cur.copy("COPY raw_events (...) FROM STDIN")`** — протокол bulk-загрузки PostgreSQL (подробный разбор — в разделе 2).
- **`conn.commit()` внутри цикла** — транзакция на чанк.

⚠️ **Опасность чанкованной дедупликации:** дубликат может оказаться в соседнем чанке — тогда `drop_duplicates` его не поймает. Полную гарантию даёт уникальный индекс + `ON CONFLICT` на стороне БД (мостик к семинару 12).


In [ ]:
# ДЕМО: превью данных и transform на маленьком примере

# 1. Превью первых строк CSV
preview = pd.read_csv(CSV, nrows=5)
preview


In [ ]:
# ДЕМО: transform + загрузка через COPY на маленьком примере

DTYPES = {"device_id": "int32", "region": "string",
          "event_type": "string", "status": "string"}

def transform(df: pd.DataFrame) -> pd.DataFrame:
    # шаг 1: убрать строки с пропусками в ключевых полях
    df = df.dropna(subset=["value", "region"])
    # шаг 2: дедупликация по бизнес-ключу
    return df.drop_duplicates(subset=["event_id"], keep="first")

# Демонстрация transform на 1000 строках
sample = pd.read_csv(CSV, nrows=1000, dtype=DTYPES, parse_dates=["ts"])
print("до transform :", len(sample))
clean = transform(sample)
print("после        :", len(clean))

# Демонстрация cur.copy() на 3 строках
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("TRUNCATE raw_events")
    conn.commit()

    with conn.cursor() as cur, cur.copy(
        "COPY raw_events (event_id, ts, device_id, region,"
        " event_type, value, status) FROM STDIN"
    ) as copy:
        for row in clean.head(3).itertuples(index=False):
            copy.write_row(row)
    conn.commit()

    with conn.cursor() as cur:
        cur.execute("SELECT event_id, ts, device_id, region, event_type, value, status"
                    " FROM raw_events")
        for row in cur.fetchall():
            print(row)


### Упражнение 1. Соберите ETL-пайплайн целиком

**Задание.** Прочитайте `events.csv` чанками по 100 000 строк, примените `transform()` к каждому чанку и загрузите очищенные строки в `raw_events` через `cur.copy()`. Замерьте время каждой фазы (`extract`, `transform`, `load`) и их доли.

**Шаги.**
1. Перед началом: `TRUNCATE raw_events`.
2. Итерируйтесь по `pd.read_csv(CSV, chunksize=100_000, dtype=DTYPES, parse_dates=["ts"])`.
3. Для каждого чанка: `transform(chunk)` → `cur.copy(...)` с `copy.write_row(row)` для каждой строки.
4. `conn.commit()` **внутри** цикла (транзакция на чанк).
5. Замеряйте время на каждой фазе через `time.perf_counter()`.
6. Напечатайте: сколько строк прочитано, сколько загружено, отброшено; тайминги и доли.

**Ориентир.** Счётчики строк воспроизводятся точно на любом железе: **1 000 000** прочитано, **984 023** загружено, **15 977** отброшено. Абсолютные тайминги зависят от машины, но **доли стабильны**: load ≈ 75–85% времени, transform — единицы процентов. Если у вас transform сопоставим с load — почти наверняка очистка написана циклом `iterrows()`, а не векторными `dropna`/`drop_duplicates`.

**На подумать (обсудим после решения).**
1. Что произойдёт с памятью, если убрать `chunksize` и файл будет 50 ГБ?
2. Почему `conn.commit()` стоит *внутри* цикла, а не после него?
3. Где в коде скрыта потеря данных, если файл отсортирован так, что дубликаты попадают в разные чанки?


In [7]:
# УПРАЖНЕНИЕ 1
import time

DTYPES = {"device_id": "int32", "region": "string",
          "event_type": "string", "status": "string"}

def transform(df: pd.DataFrame) -> pd.DataFrame:
    # шаг 1: убрать строки с пропусками в value, region
    # шаг 2: дедупликация по event_id
    # TODO: реализовать
    return df

# TODO: TRUNCATE raw_events

# TODO: замерить extract/transform/load по чанкам
# t_extract_total = 0.0
# t_transform_total = 0.0
# t_load_total = 0.0
# read_rows = 0
# loaded_rows = 0

# for chunk in pd.read_csv(CSV, chunksize=100_000, dtype=DTYPES, parse_dates=["ts"]):
#     ... замер extract, transform, load ...

# TODO: напечатать итог: прочитано / загружено / отброшено, тайминги и доли


## 2. Best practices батчинга

### 2.1. Что происходит при INSERT (мини-теория)

Каждый `INSERT` через `execute()` — это полный сетевой round-trip: клиент сериализует запрос → сервер парсит SQL, строит план, выполняет, пишет в WAL → ответ клиенту. При 1 млн строк — 1 млн round-trip'ов. Задержка сети ~0,1–1 мс доминирует над собственно записью.

**Главный принцип:** амортизировать стоимость протокола на пачке строк.

Способы (по убыванию «сетевой цены» строки):

| Способ | Что делает | Один round-trip на |
|---|---|---|
| `cur.execute` | полный round-trip | 1 строку |
| `cur.executemany` | склеивает в многострочный `INSERT ... VALUES (...), (...), ...` | пачку |
| `conn.pipeline()` | конвейер: клиент отправляет запросы, не дожидаясь ответов | пачку |
| `cur.copy("COPY ... FROM STDIN")` | потоковый bulk-протокол: сервер читает поток строк без построения плана на каждую, пишет страницами напрямую в heap | весь COPY |

Бонус: серверный `COPY FROM '/path/events.csv'` — файл уже в контейнере, данные вообще не покидают машину СУБД. Используется при загрузке дампов.

> В psycopg 3.1 экспериментальный `cursor.batch()` помечен устаревшим, в 3.3 удалён; вместо него — pipeline-режим `conn.pipeline()` (https://www.psycopg.org/psycopg3/docs/advanced/pipeline.html).


### 2.2. Упражнение 2. «Гонка» способов вставки

**Задание.** Сравните четыре способа вставки на **20 000** синтетических строк: построчный `execute`, `executemany`, pipeline и `COPY FROM STDIN`. Таблицу `TRUNCATE`-ите перед каждым прогоном, чтобы все способы стартовали с пустой таблицы.

**Шаги.**
1. Создайте таблицу `bench_race (id INT, name TEXT, val INT)` (без PK — PK добавит накладные расходы).
2. Сгенерируйте `rows` — список из 20 000 кортежей `(id, name, val)`.
3. Реализуйте 4 функции замера. Каждая: `TRUNCATE` → вставка → `commit` → замер времени.
4. Напечатайте таблицу: способ, время, строк/с, ускорение относительно построчного.
5. Сделайте вывод: во сколько раз `COPY` быстрее построчного `execute`?

**Ориентир.** Ранжирование стабильно на любом железе:
- `execute` построчно — 1×
- `executemany` — ~10×
- `pipeline` — ~10× (на localhost может быть медленнее `executemany` — выигрыш pipeline растёт с сетевой задержкой)
- `COPY` — **~100×**

**Подсказки.**
- `executemany`: `cur.executemany("INSERT INTO bench_race VALUES (%s, %s, %s)", rows)`
- `COPY`: `with cur.copy("COPY bench_race FROM STDIN") as copy: for r in rows: copy.write_row(r)`
- pipeline: `with conn.pipeline(): ...` — конвейерный режим psycopg3

**На подумать.** Почему pipeline не всегда быстрее `executemany`?


In [8]:
# УПРАЖНЕНИЕ 2
import time

N = 20_000
rows_race = [(i, f"item_{i}", i % 100) for i in range(1, N + 1)]

# Создать таблицу bench_race
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS bench_race")
        cur.execute("CREATE TABLE bench_race (id INT, name TEXT, val INT)")
    conn.commit()

def time_insert(insert_fn):
    """TRUNCATE → insert_fn(conn) → возвращает время."""
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            cur.execute("TRUNCATE bench_race")
        conn.commit()
        t0 = time.perf_counter()
        insert_fn(conn)
        return time.perf_counter() - t0

def insert_loop(conn):
    # TODO: for r in rows_race: cur.execute("INSERT INTO bench_race VALUES (%s, %s, %s)", r)
    pass

def insert_executemany(conn):
    # TODO: cur.executemany("INSERT INTO bench_race VALUES (%s, %s, %s)", rows_race)
    pass

def insert_pipeline(conn):
    # TODO: with conn.pipeline(): ...
    pass

def insert_copy(conn):
    # TODO: with cur.copy("COPY bench_race FROM STDIN") as copy:
    #           for r in rows_race: copy.write_row(r)
    pass

# TODO: замерьте все 4 и напечатайте таблицу (время, строк/с, ускорение)


## 3. Транзакция на батч vs одна большая транзакция

| Стратегия | Плюсы | Минусы |
|---|---|---|
| **Коммит на каждый чанк** | При обрыве загруженное сохранено; короткие блокировки; меньше WAL «в полёте» | Нужна идемпотентность для перезапуска (семинар 12) |
| **Одна транзакция на весь файл** | Атомарность «всё или ничего» | При обрыве на 99% теряется всё; длинная транзакция держит блокировки и мешает autovacuum |

Промышленный стандарт — **транзакция на батч + журнал загрузки** (семинар 12).

**WAL** (Write-Ahead Log) сбрасывается на диск при `commit()`. Много мелких коммитов = много fsync = медленнее. Один большой коммит = один fsync, но вся транзакция «висит» в памяти и блокирует autovacuum.


### Упражнение 3. Сравните стратегии транзакций

**Задание.** Загрузите **100 000** строк в таблицу `bench_tx (id INT, name TEXT)` двумя способами и сравните время:

1. **Один коммит в конце:** открыть соединение, `executemany` всех строк, `commit()` один раз.
2. **Коммит на чанк:** разбить `rows` на чанки по 10 000 и после каждого чанка делать `commit()`.

**Шаги.**
1. Создайте `bench_tx (id INT, name TEXT)`.
2. Сгенерируйте 100 000 кортежей `(i, f"row_{i}")`.
3. Замерьте `single_commit`: `executemany` + один `commit()`.
4. Замерьте `chunk_commit`: 10 чанков по 10 000 + `commit()` после каждого.
5. Напечатайте оба времени и разницу.
6. Сделайте вывод: какая стратегия быстрее и почему? Какой риск у каждой?

**Ориентир.** На 100 000 строк `single_commit` обычно быстрее на 10–30% (меньше fsync). Если у вас разница <5% — проверьте, что `TRUNCATE` и `commit` не попали в замер.

**Критерий успеха.** Оба замера напечатаны, вывод сформулирован.


In [9]:
# УПРАЖНЕНИЕ 3
import time

N = 100_000
CHUNK = 10_000
rows_tx = [(i, f"row_{i}") for i in range(1, N + 1)]

# Создать таблицу
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS bench_tx")
        cur.execute("CREATE TABLE bench_tx (id INT, name TEXT)")
    conn.commit()

# TODO: Замер 1 — один коммит в конце
#   TRUNCATE, t0, executemany(все rows_tx), commit, t1

# TODO: Замер 2 — коммит на чанк по 10_000
#   TRUNCATE, t0, цикл: executemany(кусок), commit; t1

# TODO: напечатать оба времени и разницу

# (Дополнительно, по желанию) проверка отката при ошибке:
#   - после TRUNCATE: ALTER TABLE bench_tx ADD PRIMARY KEY (id)
#   - в одном из чанков подложить дубликат id (например, id=1)
#   - поймать UniqueViolation в обоих режимах
#   - SELECT count(*) FROM bench_tx — сколько строк осталось?


## 4. Влияние индексов и UNLOGGED-таблиц на скорость загрузки

### Индексы

Каждый `INSERT`/`COPY` обновляет **все индексы** таблицы. Массовая загрузка с индексами может быть в разы медленнее, чем без них. Официальная рекомендация PostgreSQL («Populating a Database», https://www.postgresql.org/docs/16/populate.html): при массовой загрузке индексы **удаляют перед загрузкой и пересоздают после** + `ANALYZE`.

### UNLOGGED-таблицы

`CREATE UNLOGGED TABLE` не пишет WAL (Write-Ahead Log). Это ускоряет загрузку на 20–40%, но:

- ⚠️ Таблица **обнуляется** при сбое сервера.
- ⚠️ Таблица **не реплицируется** на реплики.

Приемлемо для **staging**, недопустимо для «золотых» данных.


### Упражнение 4. Замер влияния индекса и UNLOGGED

**Задание.** На таблице `bench_idx (id INT, device_id INT, value NUMERIC)` измерьте скорость COPY при трёх конфигурациях.

**Шаги.**
1. Создайте `bench_idx` как обычную (LOGGED) таблицу **без индексов**.
2. Сгенерируйте 100 000 строк `(id, device_id, value)` (`device_id` — 1..500, `value` — случайное).
3. **Замер A (baseline):** `COPY` 100 000 строк, засеките время.
4. Создайте индекс: `CREATE INDEX ON bench_idx (device_id);`
5. `TRUNCATE` и **Замер B (с индексом):** повторите `COPY`, засеките время.
6. Удалите индекс и переведите таблицу в `UNLOGGED`: `ALTER TABLE bench_idx SET UNLOGGED;`
7. `TRUNCATE` и **Замер C (UNLOGGED):** повторите `COPY`, засеките время.
8. Напечатайте таблицу: A, B, C — время и относительные доли.
9. Сделайте вывод: как индекс и UNLOGGED влияют на время COPY.

**Ориентир.**
- B (с индексом) должен быть **медленнее** A — каждая вставленная страница обновляет ещё и B-дерево.
- C (UNLOGGED) должен быть **быстрее** A — нет записи WAL.

**Проверка типа таблицы:** `SELECT relpersistence FROM pg_class WHERE relname = 'bench_idx';` — `'u'` (unlogged) или `'p'` (обычная).

**Критерий успеха.** Три числа в таблице; вывод сформулирован: индекс → медленнее, UNLOGGED → быстрее.


In [10]:
# УПРАЖНЕНИЕ 4
import time, random

N = 100_000
rows_idx = [(i, random.randint(1, 500), round(random.random() * 1000, 2))
            for i in range(1, N + 1)]

# --- Замер A: обычная LOGGED таблица без индексов ---
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS bench_idx")
        cur.execute("CREATE TABLE bench_idx (id INT, device_id INT, value NUMERIC)")
    conn.commit()

# TODO: замер A — TRUNCATE, t0, COPY rows_idx, t1

# --- Замер B: с индексом на device_id ---
# TODO: CREATE INDEX ON bench_idx (device_id); TRUNCATE, COPY, замер B

# --- Замер C: UNLOGGED ---
# TODO: DROP INDEX, ALTER TABLE bench_idx SET UNLOGGED; TRUNCATE, COPY, замер C

# TODO: напечатать итоговую таблицу (A, B, C — время и относительные доли)

# (Восстановление после эксперимента, если планируете запускать другие ячейки)
# ALTER TABLE bench_idx SET LOGGED;  -- если переключали в UNLOGGED


## 5. Чек-лист батчинга

1. Никогда не вставлять построчно через `execute()` — минимум `executemany`, лучше `COPY`.
2. Размер батча **10k–100k** строк (совпадает с правилом ClickHouse из лекции СУБД_7).
3. **Транзакция на батч**, не на строку и не на весь файл.
4. Перед массовой загрузкой — убрать индексы/триггеры; после — пересоздать + `ANALYZE`.
5. Staging-таблица — `UNLOGGED`; целевая — обычная.
6. Загрузка должна быть **идемпотентной**: повторный прогон не порождает дублей (бизнес-ключ + `ON CONFLICT`).
7. Замерять и журналировать: сколько строк, за сколько, с каким результатом.


## Итоги и мостик к семинару 12

- **ETL** = Extract → Transform → Load; **ELT** переносит трансформации внутрь хранилища.
- **Чанкование** превращает пайплайн в потоковый: память `O(chunk)`, а не `O(dataset)`.
- Способ вставки меняет throughput на **два порядка**: ~1 тыс. → ~100 тыс. строк/с.
- Узкое место — не pandas, а **протокол вставки** и запись WAL/индексов.
- Нерешённые проблемы нашего пайплайна: дубли между чанками, обрыв посередине, отсутствие контроля «источник = приёмник». Это — семинар 12.

**Вопросы на выход.**
1. Почему `COPY` быстрее многострочного `INSERT`, хотя данные те же?
2. Чем рискует `UNLOGGED`-таблица и где её использование оправдано?
3. Почему при массовой загрузке удаляют индексы, хотя они «ускоряют базу»?
4. Что делать с дубликатами, которые попали в разные чанки?


## Ресурсы

- pandas IO tools (read_csv, chunksize): https://pandas.pydata.org/docs/user_guide/io.html
- psycopg 3 — Copy: https://www.psycopg.org/psycopg3/docs/basic/copy.html
- psycopg 3 — Pipeline: https://www.psycopg.org/psycopg3/docs/advanced/pipeline.html
- PostgreSQL COPY: https://www.postgresql.org/docs/16/sql-copy.html
- PostgreSQL «Populating a Database»: https://www.postgresql.org/docs/16/populate.html
- DataTalksClub DE Zoomcamp, Module 1: https://github.com/DataTalksClub/data-engineering-zoomcamp
